In [1]:
import pyEDM
import pandas as pd
import numpy as np
import random

p22 = pd.read_csv('./data/processed_22.csv')
p23 = pd.read_csv('./data/processed_23.csv')
p24 = pd.read_csv('./data/processed_24.csv')
p25 = pd.read_csv('./data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

Exponentially Weighted Moving

In [2]:
s = combined[["date", "Lpoly_expected_ml"]].copy()
s["date"] = pd.to_datetime(s["date"].astype(str), format="%Y%m%d")
s = s.sort_values("date").set_index("date")

# force daily frequency
s = s.asfreq("D")

ema = s["Lpoly_expected_ml"].ewm(span=30, adjust=False).mean()

s["Lpoly_filled"] = s["Lpoly_expected_ml"]
s.loc[s["Lpoly_filled"].isna(), "Lpoly_filled"] = ema

s["was_imputed"] = s["Lpoly_expected_ml"].isna()

y = s["Lpoly_expected_ml"]
y_norm = (y - np.mean(y)) / np.std(y)

df = pd.DataFrame({
    "t": np.arange(1, len(y_norm) + 1),        # numeric EDM time
    "x": y_norm
})

In [8]:
N = len(df)

res = pyEDM.Simplex(
    dataFrame=df,
    lib=f"1 {N}",        # use entire series as library
    pred=f"1 {N}",       # predict entire series (leave-one-out)
    E=4,
    tau=1,
    Tp=1,
    columns="x",
    target="x"
)

obs = res["Observations"].to_numpy()
pred = res["Predictions"].to_numpy()

mask = np.isfinite(obs) & np.isfinite(pred)

obs = obs[mask]
pred = pred[mask]

rho = np.corrcoef(obs, pred)[0, 1]
rmse = np.sqrt(np.mean((obs - pred)**2))

print("Simplex rho:", rho)
print("Simplex RMSE:", rmse)


Simplex rho: 0.9499513732894488
Simplex RMSE: 0.48987432511696716


In [9]:
def simplex_loocv(df, E=4, Tp=1):
    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        lib=f"1 {N}",
        pred=f"1 {N}",
        E=E,
        tau=1,
        Tp=Tp,
        columns="x",
        target="x"
    )
    # res columns typically: Time, Observations, Predictions (names can vary slightly)
    obs = res["Observations"].to_numpy()
    predv = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(predv)
    obs = obs[mask]
    predv = predv[mask]


    rho = np.corrcoef(obs, predv)[0, 1]
    return rho, res

def acf_at_lag(x, lag):
    # x should be 1D array
    s = pd.Series(x)
    return float(s.autocorr(lag=lag))

def random_blocks(N, block_len=250, k_blocks=3, seed=None):
    """
    Returns a pyEDM interval string with k_blocks random contiguous blocks.
    Format: "start1 end1 start2 end2 ..."
    """
    if seed is not None:
        random.seed(seed)

    max_start = N - block_len + 1
    if max_start < 1:
        raise ValueError("Time series shorter than block_len")

    starts = random.sample(range(1, max_start + 1), k=min(k_blocks, max_start))
    starts.sort()

    parts = []
    for s in starts:
        e = s + block_len - 1
        parts.extend([str(s), str(e)])
    return " ".join(parts)

def simplex_bootstrap(df, E=4, Tp=1, B=200, block_len=250, k_blocks=3):
    N = len(df)

    # EDM-valid index range
    max_i = N - Tp - (E - 1)
    if max_i <= block_len:
        return [], np.nan

    x = df["x"].to_numpy()
    acf = abs(acf_at_lag(x, Tp))

    rhos = []
    rho_eff = []

    for b in range(B):
        blocks = random_blocks(max_i, block_len=block_len, k_blocks=k_blocks)
        lib  = blocks
        pred = blocks

        res = pyEDM.Simplex(
            dataFrame=df,
            lib=lib,
            pred=pred,
            E=E,
            tau=1,
            Tp=Tp,
            columns="x",
            target="x"
        )

        obs = res["Observations"].to_numpy()
        predv = res["Predictions"].to_numpy()

        if len(obs) < 5 or np.std(obs) == 0 or np.std(predv) == 0:
            continue

        rho = np.corrcoef(obs, predv)[0, 1]
        rhos.append(rho)
        rho_eff.append(rho - acf)

    return np.array(rhos), np.array(rho_eff)


In [10]:
results = []

for tp in range(1, 31):
    rhos, rho_eff = simplex_bootstrap(df, E=4, Tp=tp, B=200, block_len=500, k_blocks=1)

    results.append({
        "Tp": tp,
        "rho_mean": np.nanmean(rhos),
        "rho_ci2.5": np.nanpercentile(rhos, 2.5),
        "rho_ci97.5": np.nanpercentile(rhos, 97.5),
        "rho_eff_mean": np.nanmean(rho_eff),
        "rho_eff_ci2.5": np.nanpercentile(rho_eff, 2.5),
        "rho_eff_ci97.5": np.nanpercentile(rho_eff, 97.5),
        "n_boot_used": len(rhos),
    })

summary = pd.DataFrame(results)
summary

C:\Users\Ourple\AppData\Local\Temp\ipykernel_12712\3232692720.py:8: RuntimeWarning: Mean of empty slice
  "rho_mean": np.nanmean(rhos),
c:\Users\Ourple\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\Ourple\AppData\Local\Temp\ipykernel_12712\3232692720.py:11: RuntimeWarning: Mean of empty slice
  "rho_eff_mean": np.nanmean(rho_eff),
C:\Users\Ourple\AppData\Local\Temp\ipykernel_12712\3232692720.py:8: RuntimeWarning: Mean of empty slice
  "rho_mean": np.nanmean(rhos),
c:\Users\Ourple\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\Ourple\AppData\Local\Temp\ipykernel_12712\3232692720.py:11: RuntimeWarning: Mean of empty slice
  "rho_eff_mean": np.nanmean(rho_eff),
C:\Users\Ourple\AppData\Local\Temp\ipykernel_12712\3232692720.py:8: RuntimeWar

,Tp,rho_mean,rho_ci2.5,rho_ci97.5,rho_eff_mean,rho_eff_ci2.5,rho_eff_ci97.5,n_boot_used
0,1,NaN,NaN,NaN,NaN,NaN,NaN,200
1,2,NaN,NaN,NaN,NaN,NaN,NaN,200
2,3,NaN,NaN,NaN,NaN,NaN,NaN,200
3,4,NaN,NaN,NaN,NaN,NaN,NaN,200
4,5,NaN,NaN,NaN,NaN,NaN,NaN,200
5,6,NaN,NaN,NaN,NaN,NaN,NaN,200
6,7,NaN,NaN,NaN,NaN,NaN,NaN,200
7,8,NaN,NaN,NaN,NaN,NaN,NaN,200
8,9,NaN,NaN,NaN,NaN,NaN,NaN,200
9,10,NaN,NaN,NaN,NaN,NaN,NaN,200


In [7]:
def debug_simplex_once(df, E=4, Tp=1, block_len=250, k_blocks=3):
    N = len(df)
    lib = random_blocks(N, block_len=block_len, k_blocks=k_blocks)
    pred = random_blocks(N, block_len=block_len, k_blocks=k_blocks)

    res = pyEDM.Simplex(
        dataFrame=df,
        lib=lib,
        pred=pred,
        E=E,
        tau=1,
        Tp=Tp,
        columns="x",
        target="x"
    )

    obs = res["Observations"].to_numpy()
    predv = res["Predictions"].to_numpy()

    print("len(obs):", len(obs))
    print("std(obs):", np.std(obs))
    print("std(pred):", np.std(predv))

    return obs, predv

obs, predv = debug_simplex_once(df, Tp=1)

len(obs): 753
std(obs): nan
std(pred): nan
